# JobLens AI: Job Matching and Skill Extraction Pipeline

This notebook combines resume–job matching evaluation with job-description skill extraction. Run the sections in order; each section can also be used independently after its dependencies are installed.

## Setup

In [ ]:
# Run this once in Google Colab.
# !pip install -q sentence-transformers datasets kagglehub pandas scikit-learn transformers torch spacy matplotlib
# !python -m spacy download en_core_web_sm -q

## Part 1: Resume–Job Matching

In [ ]:
from __future__ import annotations

import gc
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from datasets import load_dataset
from kagglehub import KaggleDatasetAdapter
import kagglehub
from sentence_transformers import (
    InputExample, SentenceTransformer, evaluation, losses
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import DataLoader

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    base_model_name: str = 'all-MiniLM-L6-v2'
    drive_model_path: str = '/content/drive/MyDrive/joblens-finetuned-sbert'
    local_model_path: str = 'joblens-finetuned-sbert'
    max_jobs: int = 50
    max_resumes: int = 100
    top_k: int = 3
    batch_size: int = 16
    epochs: int = 3
    device: str = 'cpu'


CONFIG = PipelineConfig()

# Resume Atlas category -> keyword expected in a LinkedIn posting title.
JOB_CATEGORY_KEYWORDS = {
    'Accountant': 'accountant', 'Advocate': 'advocate', 'Agriculture': 'agricultur',
    'Apparel': 'apparel', 'Architecture': 'architect', 'Arts': 'arts',
    'Automobile': 'automobil', 'Aviation': 'aviation', 'Banking': 'banking',
    'Blockchain': 'blockchain', 'BPO': 'bpo',
    'Building and Construction': 'construction', 'Business Analyst': 'business analyst',
    'Civil Engineer': 'civil engineer', 'Consultant': 'consultant',
    'Data Science': 'data scien', 'Database': 'database', 'Designing': 'design',
    'DevOps': 'devops', 'Digital Media': 'digital media',
    'DotNet Developer': 'dotnet', 'Education': 'education',
    'Electrical Engineering': 'electrical', 'ETL Developer': 'etl',
    'Finance': 'financ', 'Food and Beverages': 'food',
    'Health and Fitness': 'health', 'Human Resources': 'human resource',
    'Information Technology': 'information technolog', 'Java Developer': 'java',
    'Management': 'management', 'Mechanical Engineer': 'mechanical',
    'Network Security Engineer': 'network security',
    'Operations Manager': 'operations', 'PMO': 'pmo',
    'Public Relations': 'public relation', 'Python Developer': 'python',
    'React Developer': 'react', 'Sales': 'sales', 'SAP Developer': 'sap',
    'SQL Developer': 'sql', 'Testing': 'test', 'Web Designing': 'web design',
}


### Data preparation functions

These functions load and clean the data, then construct the ground truth.

In [ ]:
def load_resume_dataset() -> pd.DataFrame:
    """Convert Hugging Face Resume Atlas into the standard resume DataFrame."""
    raw_resumes = load_dataset('ahmedheakl/resume-atlas')['train'].to_pandas()
    return pd.DataFrame({
        'id': [f'R{i}' for i in range(len(raw_resumes))],
        'category': raw_resumes['Category'].fillna('').str.strip(),
        'resume_text': raw_resumes['Text'].fillna('').astype(str),
    })


def load_job_postings() -> pd.DataFrame:
    """Convert Kaggle LinkedIn postings into the standard job DataFrame."""
    raw_jobs = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS, 'arshkon/linkedin-job-postings', 'postings.csv'
    )
    return pd.DataFrame({
        'id': [f'J{i}' for i in range(len(raw_jobs))],
        'title': raw_jobs['title'].fillna('').astype(str),
        'job_description': raw_jobs['description'].fillna('').astype(str),
    })


def infer_job_category(job_title: str) -> str | None:
    """Infer the resume category for evaluation from title keywords."""
    normalized_title = str(job_title).lower()
    return next((category for category, keyword in JOB_CATEGORY_KEYWORDS.items()
                 if keyword in normalized_title), None)


def prepare_evaluation_data(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build a small reproducible evaluation set with a known relevant resume for each category."""
    tagged_jobs = jobs.copy()
    tagged_jobs['matched_category'] = tagged_jobs['title'].map(infer_job_category)
    tagged_jobs = tagged_jobs.dropna(subset=['matched_category'])

    category_to_resume_id = (
        resumes.groupby('category', group_keys=False).apply(
            lambda group: group.sample(n=1, random_state=RANDOM_SEED)
        ).set_index('category')['id'].to_dict()
    )
    tagged_jobs['correct_resume_id'] = tagged_jobs['matched_category'].map(category_to_resume_id)
    tagged_jobs = (tagged_jobs.dropna(subset=['correct_resume_id'])
                   .sample(n=min(config.max_jobs, len(tagged_jobs)), random_state=RANDOM_SEED)
                   .reset_index(drop=True))

    required_ids = set(tagged_jobs['correct_resume_id'])
    required_resumes = resumes[resumes['id'].isin(required_ids)]
    optional_resumes = resumes[~resumes['id'].isin(required_ids)]
    remaining_count = max(0, config.max_resumes - len(required_resumes))
    sampled_optional = optional_resumes.sample(
        n=min(remaining_count, len(optional_resumes)), random_state=RANDOM_SEED
    )
    evaluation_resumes = pd.concat([required_resumes, sampled_optional], ignore_index=True)
    return evaluation_resumes, tagged_jobs


### Matching and evaluation functions

These functions create embeddings and similarity matrices and calculate Precision@K and NDCG@K.

In [ ]:
def build_tfidf_similarity_matrix(resumes: pd.DataFrame, jobs: pd.DataFrame) -> np.ndarray:
    """TF-IDF cosine similarity matrix: (job count, resume count)."""
    corpus = resumes['resume_text'].tolist() + jobs['job_description'].tolist()
    vectors = TfidfVectorizer(stop_words='english').fit_transform(corpus)
    return cosine_similarity(vectors[len(resumes):], vectors[:len(resumes)])


def build_embedding_similarity_matrix(
    model: SentenceTransformer, resumes: pd.DataFrame, jobs: pd.DataFrame, batch_size: int
) -> np.ndarray:
    """Create a job–resume cosine-similarity matrix using SentenceTransformer."""
    resume_embeddings = model.encode(
        resumes['resume_text'].tolist(), batch_size=batch_size, show_progress_bar=False
    )
    job_embeddings = model.encode(
        jobs['job_description'].tolist(), batch_size=batch_size, show_progress_bar=False
    )
    return cosine_similarity(job_embeddings, resume_embeddings)


def calculate_ranking_metrics(
    similarity_matrix: np.ndarray, jobs: pd.DataFrame, resumes: pd.DataFrame, top_k: int
) -> Dict[str, object]:
    """Return Precision@K, NDCG@K, and ranking details for one relevant resume per job."""
    resume_ids = resumes['id'].tolist()
    hits, ndcg_scores, details = 0, [], []
    for row_index, job in jobs.iterrows():
        ranked_indices = np.argsort(similarity_matrix[row_index])[::-1]
        ranked_ids = [resume_ids[index] for index in ranked_indices]
        correct_id = job['correct_resume_id']
        rank = ranked_ids.index(correct_id) + 1 if correct_id in ranked_ids else None
        hit = rank is not None and rank <= top_k
        hits += int(hit)
        ndcg_scores.append(1 / np.log2(rank + 1) if hit else 0.0)
        details.append({
            'job_title': job['title'], 'correct_resume': correct_id,
            'top1_predicted': ranked_ids[0],
            'top1_score': round(float(similarity_matrix[row_index, ranked_indices[0]]), 3),
            f'hit_at_{top_k}': hit,
        })
    return {
        f'Precision@{top_k}': round(hits / len(jobs), 3),
        f'NDCG@{top_k}': round(float(np.mean(ndcg_scores)), 3),
        'details': pd.DataFrame(details),
    }


def evaluate_similarity_model(
    model_name: str, similarity_matrix: np.ndarray, jobs: pd.DataFrame, resumes: pd.DataFrame, top_k: int
) -> Dict[str, object]:
    metrics = calculate_ranking_metrics(similarity_matrix, jobs, resumes, top_k)
    return {'Model': model_name, **metrics}


### Fine-tuned model management

The model is loaded from Drive when available. Otherwise, it is trained and saved locally and to Drive.

In [ ]:
def mount_google_drive() -> bool:
    """Mount Google Drive only in Colab and return whether it is available."""
    try:
        from google.colab import drive
    except ImportError:
        print('Google Drive is unavailable outside Google Colab; skipping Drive.')
        return False
    drive.mount('/content/drive', force_remount=False)
    return True


def model_files_exist(model_path: str | Path) -> bool:
    """Check whether a directory contains the minimum SentenceTransformer model files."""
    path = Path(model_path)
    return path.is_dir() and (path / 'config_sentence_transformers.json').exists()


def create_training_examples(resumes: pd.DataFrame, jobs: pd.DataFrame) -> List[InputExample]:
    """Create positive pairs from the same job category and negative pairs from other categories."""
    resumes_by_category = resumes.groupby('category')['resume_text'].apply(list).to_dict()
    examples: List[InputExample] = []
    rng = random.Random(RANDOM_SEED)
    for _, job in jobs.iterrows():
        category = job['matched_category']
        positives = resumes_by_category.get(category, [])
        if not positives:
            continue
        job_text = job['job_description'][:512]
        for resume_text in rng.sample(positives, k=min(2, len(positives))):
            examples.append(InputExample(texts=[job_text, resume_text[:512]], label=1.0))
        other_categories = [key for key in resumes_by_category if key != category]
        for negative_category in rng.sample(other_categories, k=min(4, len(other_categories))):
            negative_resume = rng.choice(resumes_by_category[negative_category])
            examples.append(InputExample(texts=[job_text, negative_resume[:512]], label=0.0))
    rng.shuffle(examples)
    if len(examples) < 2:
        raise ValueError('There are not enough training pairs for fine-tuning.')
    return examples


def train_finetuned_model(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> SentenceTransformer:
    """Fine-tune the base model and save the best model to `local_model_path`."""
    examples = create_training_examples(resumes, jobs)
    split_index = max(1, int(len(examples) * 0.9))
    train_examples, validation_examples = examples[:split_index], examples[split_index:]
    model = SentenceTransformer(config.base_model_name, device=config.device)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=config.batch_size)
    evaluator = None
    if validation_examples:
        evaluator = evaluation.EmbeddingSimilarityEvaluator(
            [item.texts[0] for item in validation_examples],
            [item.texts[1] for item in validation_examples],
            [item.label for item in validation_examples], name='validation'
        )
    model.fit(
        train_objectives=[(train_loader, losses.CosineSimilarityLoss(model))],
        evaluator=evaluator, epochs=config.epochs,
        warmup_steps=max(1, int(len(train_loader) * 0.1)),
        evaluation_steps=max(1, int(len(train_loader) * 0.5)),
        output_path=config.local_model_path, save_best_model=True, show_progress_bar=True,
    )
    return SentenceTransformer(config.local_model_path, device=config.device)


def load_or_train_finetuned_model(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> Tuple[SentenceTransformer, str]:
    """Obtain the model in this order: Drive, local cache, then fine-tuning."""
    drive_is_available = mount_google_drive()
    if drive_is_available and model_files_exist(config.drive_model_path):
        print(f'✅ Loaded fine-tuned model from Google Drive: {config.drive_model_path}')
        return SentenceTransformer(config.drive_model_path, device=config.device), 'Google Drive'
    if model_files_exist(config.local_model_path):
        print(f'✅ Loaded fine-tuned model from local cache: {config.local_model_path}')
        return SentenceTransformer(config.local_model_path, device=config.device), 'local cache'

    print('ℹ️ No saved fine-tuned model was found; training now.')
    model = train_finetuned_model(resumes, jobs, config)
    if drive_is_available:
        destination = Path(config.drive_model_path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(config.local_model_path, destination, dirs_exist_ok=True)
        print(f'✅ Saved trained model to Google Drive: {destination}')
    return model, 'trained this run'


### Run All pipeline

The final cell below runs every step in dependency order. Adjust only `PipelineConfig` when needed.

In [ ]:
def evaluate_pretrained_models(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> List[Dict[str, object]]:
    """Collect results for TF-IDF and pretrained SBERT models."""
    results = [evaluate_similarity_model(
        'TF-IDF (Baseline)', build_tfidf_similarity_matrix(resumes, jobs),
        jobs, resumes, config.top_k
    )]
    model_ids = ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'BAAI/bge-base-en-v1.5']
    for model_id in model_ids:
        print(f'Evaluating: {model_id}')
        model = SentenceTransformer(model_id, device=config.device)
        similarity = build_embedding_similarity_matrix(model, resumes, jobs, config.batch_size)
        results.append(evaluate_similarity_model(model_id, similarity, jobs, resumes, config.top_k))
        del model, similarity
        gc.collect()
    return results


def run_job_matching_pipeline(config: PipelineConfig = CONFIG) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run All entry point: load → prepare → load or train → evaluate → save."""
    print('1/5 Loading resume and job-posting data')
    all_resumes = load_resume_dataset()
    all_jobs = load_job_postings()

    print('2/5 Preparing evaluation data')
    resumes, jobs = prepare_evaluation_data(all_resumes, all_jobs, config)
    print(f'   resumes={len(resumes)}, jobs={len(jobs)}')

    print('3/5 Obtaining the fine-tuned model (Google Drive first)')
    finetuned_model, model_source = load_or_train_finetuned_model(resumes, jobs, config)

    print('4/5 Evaluating baseline and pretrained models')
    results = evaluate_pretrained_models(resumes, jobs, config)

    print('5/5 Evaluating the fine-tuned model and saving results')
    finetuned_similarity = build_embedding_similarity_matrix(
        finetuned_model, resumes, jobs, config.batch_size
    )
    results.append(evaluate_similarity_model(
        f'SBERT Fine-tuned (JobLens; {model_source})', finetuned_similarity,
        jobs, resumes, config.top_k
    ))

    metric_columns = ['Model', f'Precision@{config.top_k}', f'NDCG@{config.top_k}']
    comparison = pd.DataFrame(results)[metric_columns]
    output_file = f'comparison_models_pool{len(resumes)}.csv'
    comparison.to_csv(output_file, index=False)
    print('\n' + comparison.to_string(index=False))
    print(f'\n✅ Saved comparison results: {output_file}')
    return comparison, pd.DataFrame(results)[['Model', 'details']]


# This final Run All cell calls the functions in dependency order.
comparison_df, ranking_details_df = run_job_matching_pipeline()


## Part 2: Functional Skill Extraction Pipeline

The pipeline loads job postings once, extracts BERT NER and O*NET keyword skills in one pass, then produces hybrid results, summary metrics, and visualizations.

### Configuration and model setup

In [ ]:
import time
from dataclasses import dataclass
from typing import Any

import matplotlib.pyplot as plt
from transformers import pipeline


@dataclass(frozen=True)
class SkillExtractionConfig:
    model_name: str = "GalalEwida/LLM-BERT-Model-Based-Skills-Extraction-from-jobdescription"
    sample_size: int = 2000
    confidence_threshold: float = 0.7
    max_text_length: int = 800
    progress_interval: int = 10
    output_prefix: str = "skill_extraction"


SKILL_CONFIG = SkillExtractionConfig()


ONET_SKILLS = [
    "python", "java", "javascript", "sql", "r", "c++", "scala", "kotlin",
    "react", "node.js", "spring boot", "docker", "kubernetes", "aws", "azure",
    "machine learning", "deep learning", "tensorflow", "pytorch", "nlp",
    "data analysis", "data visualization", "tableau", "power bi", "excel",
    "rest api", "microservices", "git", "linux", "devops", "ci/cd",
    "spark", "hadoop", "etl", "database", "postgresql", "mongodb",
    "project management", "agile", "scrum", "product management",
    "strategic planning", "business analysis", "stakeholder management",
    "risk management", "budgeting", "forecasting", "financial analysis",
    "accounting", "auditing", "tax", "compliance",
    "recruitment", "talent acquisition", "employee relations",
    "performance management", "training", "onboarding", "hris",
    "organizational development", "change management",
    "seo", "digital marketing", "content marketing", "social media",
    "google analytics", "crm", "salesforce", "lead generation",
    "market research", "brand management", "copywriting",
    "autocad", "solidworks", "mechanical design", "electrical engineering",
    "civil engineering", "structural analysis", "quality control",
    "lean manufacturing", "six sigma", "supply chain",
    "patient care", "clinical research", "medical coding", "hipaa",
    "electronic health records", "nursing", "pharmacy",
    "communication", "leadership", "teamwork", "problem solving",
    "critical thinking", "time management", "presentation",
    "negotiation", "customer service", "conflict resolution",
]
ONET_SKILLS_SET = sorted(set(ONET_SKILLS), key=len, reverse=True)

In [ ]:
def load_skill_extraction_jobs(config: SkillExtractionConfig) -> pd.DataFrame:
    """Load and reproducibly sample the shared LinkedIn job-posting dataset."""
    jobs = load_job_postings()
    return jobs.sample(n=min(config.sample_size, len(jobs)), random_state=RANDOM_SEED).reset_index(drop=True)


def load_skill_ner_model(config: SkillExtractionConfig) -> Any:
    """Load the JobBERT NER pipeline on CPU; set `device=0` here to use a GPU."""
    return pipeline(
        "token-classification",
        model=config.model_name,
        aggregation_strategy="simple",
        device=-1,
    )


def extract_bert_skills(text: str, ner_model: Any, config: SkillExtractionConfig) -> list[str]:
    """Extract unique, high-confidence skills from one job description using BERT NER."""
    entities = ner_model(str(text)[:config.max_text_length])
    return list(dict.fromkeys(
        entity["word"].strip().lower()
        for entity in entities
        if entity["score"] >= config.confidence_threshold and len(entity["word"].strip()) >= 2
    ))


def extract_onet_skills(text: str) -> list[str]:
    """Extract non-overlapping O*NET dictionary skills from one job description."""
    normalized_text = str(text).lower()
    matched, used_positions = [], set()
    for skill in ONET_SKILLS_SET:
        start = normalized_text.find(skill)
        if start < 0:
            continue
        positions = set(range(start, start + len(skill)))
        if positions & used_positions:
            continue
        matched.append(skill)
        used_positions.update(positions)
    return matched


def combine_skills(bert_skills: list[str], onet_skills: list[str], mode: str = "intersection") -> list[str]:
    """Combine BERT and O*NET results using intersection or union mode."""
    if mode == "union":
        return sorted(set(bert_skills) | set(onet_skills))
    if mode != "intersection":
        raise ValueError("mode must be either 'intersection' or 'union'")
    return sorted({
        onet_skill for onet_skill in onet_skills
        if any(bert_skill in onet_skill or onet_skill in bert_skill for bert_skill in bert_skills)
    })


def extract_skill_profile(
    text: str, ner_model: Any, config: SkillExtractionConfig, hybrid_mode: str = "union"
) -> dict[str, list[str]]:
    """Extract BERT, O*NET, and hybrid skills from one job description or resume.

    This is the shared Part 2 extraction interface used by both batch evaluation
    and the single resume–job analysis in Part 3.
    """
    try:
        bert_skills = extract_bert_skills(text, ner_model, config)
    except Exception as error:
        print(f"BERT skill extraction skipped: {error}")
        bert_skills = []
    onet_skills = extract_onet_skills(text)
    return {
        "bert_skills": bert_skills,
        "onet_skills": onet_skills,
        "hybrid_skills": combine_skills(bert_skills, onet_skills, mode=hybrid_mode),
    }

### Extraction, evaluation, and visualization functions

In [ ]:
def extract_skills_for_jobs(
    jobs: pd.DataFrame, ner_model: Any, config: SkillExtractionConfig, hybrid_mode: str = "intersection"
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run BERT NER and O*NET extraction once per job and create all result tables."""
    records = []
    for index, (_, job) in enumerate(jobs.iterrows(), start=1):
        skill_profile = extract_skill_profile(job["job_description"], ner_model, config, hybrid_mode)
        records.append({
            "job_id": job["id"], "job_title": job["title"],
            "bert_skills": skill_profile["bert_skills"],
            "onet_skills": skill_profile["onet_skills"],
            "hybrid_skills": skill_profile["hybrid_skills"],
            "n_bert": len(skill_profile["bert_skills"]),
            "n_onet": len(skill_profile["onet_skills"]),
            "n_hybrid": len(skill_profile["hybrid_skills"]),
        })
        if index % config.progress_interval == 0 or index == len(jobs):
            print(f"Progress: {index}/{len(jobs)} postings")

    extracted = pd.DataFrame(records)
    bert_results = extracted[["job_id", "job_title", "bert_skills", "n_bert"]].rename(
        columns={"bert_skills": "all_skills", "n_bert": "n_skills"}
    )
    onet_results = extracted[["job_id", "job_title", "onet_skills", "n_onet"]].rename(
        columns={"onet_skills": "all_skills", "n_onet": "n_skills"}
    )
    return bert_results, onet_results, extracted


def build_skill_comparison(
    bert_results: pd.DataFrame, onet_results: pd.DataFrame, hybrid_results: pd.DataFrame
) -> pd.DataFrame:
    """Calculate coverage and skill-count statistics for the three extraction methods."""
    method_counts = {
        "BERT NER": bert_results["n_skills"],
        "O*NET Keyword Matching": onet_results["n_skills"],
        "Hybrid": hybrid_results["n_hybrid"],
    }
    return pd.DataFrame({
        "Method": method_counts.keys(),
        "Coverage (%)": [round((counts > 0).mean() * 100, 1) for counts in method_counts.values()],
        "Avg Skills/Job": [round(counts.mean(), 1) for counts in method_counts.values()],
        "Max Skills/Job": [int(counts.max()) for counts in method_counts.values()],
        "Median Skills/Job": [round(counts.median(), 1) for counts in method_counts.values()],
    })


def plot_skill_comparison(
    comparison: pd.DataFrame, bert_results: pd.DataFrame, onet_results: pd.DataFrame, hybrid_results: pd.DataFrame, output_file: str
) -> None:
    """Create and save coverage, average-count, and distribution comparison charts."""
    colors = ["#2E5FA3", "#27AE60", "#E67E22"]
    labels = comparison["Method"].tolist()
    fig, axes = plt.subplots(1, 3, figsize=(17, 6))
    fig.suptitle("JobLens AI — Skill Extraction Method Comparison", fontsize=14, fontweight="bold")

    for axis, column, title, ylabel in [
        (axes[0], "Coverage (%)", "Coverage", "Coverage (%)"),
        (axes[1], "Avg Skills/Job", "Average Skills per Job", "Skills"),
    ]:
        bars = axis.bar(labels, comparison[column], color=colors)
        axis.set_title(title)
        axis.set_ylabel(ylabel)
        axis.tick_params(axis="x", rotation=20)
        for bar, value in zip(bars, comparison[column]):
            axis.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value}", ha="center", va="bottom")

    axes[2].boxplot([bert_results["n_skills"], onet_results["n_skills"], hybrid_results["n_hybrid"]], labels=labels)
    axes[2].set_title("Skill-count Distribution")
    axes[2].set_ylabel("Skills")
    axes[2].tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.show()


def run_skill_extraction_pipeline(
    config: SkillExtractionConfig = SKILL_CONFIG, hybrid_mode: str = "intersection"
) -> dict[str, pd.DataFrame]:
    """Run the complete functional skill-extraction workflow and save its artifacts."""
    print("1/4 Loading job postings")
    jobs = load_skill_extraction_jobs(config)
    print(f"   Sampled {len(jobs)} postings")

    print("2/4 Loading BERT NER model")
    ner_model = load_skill_ner_model(config)

    print("3/4 Extracting BERT NER, O*NET, and hybrid skills")
    bert_results, onet_results, hybrid_results = extract_skills_for_jobs(jobs, ner_model, config, hybrid_mode)

    print("4/4 Building comparison and saving artifacts")
    comparison = build_skill_comparison(bert_results, onet_results, hybrid_results)
    bert_results.to_csv(f"{config.output_prefix}_bert.csv", index=False)
    onet_results.to_csv(f"{config.output_prefix}_onet.csv", index=False)
    hybrid_results.to_csv(f"{config.output_prefix}_hybrid.csv", index=False)
    comparison.to_csv(f"{config.output_prefix}_comparison.csv", index=False)
    plot_skill_comparison(
        comparison, bert_results, onet_results, hybrid_results, f"{config.output_prefix}_comparison.png"
    )
    print(comparison.to_string(index=False))
    return {
        "jobs": jobs, "bert_results": bert_results, "onet_results": onet_results,
        "hybrid_results": hybrid_results, "comparison": comparison,
    }

### Run All

In [ ]:
# Run the complete Part 2 workflow.
skill_extraction_outputs = run_skill_extraction_pipeline()
skill_extraction_outputs["comparison"]

## Part 3: Single Resume–Job Analysis

Use this section with one user-provided job description and one user-provided resume. It produces an explainable JobLens Match Score and a skill-gap analysis. The score is a job-fit indicator, not a hiring prediction.

In [ ]:
@dataclass(frozen=True)
class ApplicationAnalysisConfig:
    similarity_model_name: str = "all-MiniLM-L6-v2"
    semantic_weight: float = 0.50
    skill_weight: float = 0.35
    requirement_weight: float = 0.15


APPLICATION_CONFIG = ApplicationAnalysisConfig()


# Normalize common variants before comparing skills. Extend this mapping as new variants appear.
SKILL_ALIASES = {
    "aws": "amazon web services", "amazon aws": "amazon web services",
    "gcp": "google cloud platform", "google cloud": "google cloud platform",
    "js": "javascript", "ts": "typescript", "node": "node.js",
    "postgres": "postgresql", "ms sql": "sql", "microsoft sql server": "sql",
    "powerbi": "power bi", "ml": "machine learning", "ai": "artificial intelligence",
}


def normalize_skill(skill: str) -> str:
    """Normalize a skill label to support reliable matching across common variants."""
    normalized = " ".join(str(skill).lower().replace("##", "").split())
    return SKILL_ALIASES.get(normalized, normalized)


def normalize_skills(skills: list[str]) -> list[str]:
    """Normalize, remove blank values, and preserve the first occurrence of each skill."""
    return list(dict.fromkeys(
        normalized for skill in skills
        if (normalized := normalize_skill(skill))
    ))

In [ ]:
def extract_document_skills(
    text: str, ner_model: Any, skill_config: SkillExtractionConfig
) -> dict[str, list[str]]:
    """Extract normalized BERT, O*NET, and hybrid skill lists from any text document."""
    skill_profile = extract_skill_profile(text, ner_model, skill_config, hybrid_mode="union")
    return {
        method: normalize_skills(skills)
        for method, skills in skill_profile.items()
    }


def extract_job_skills(job_description: str, ner_model: Any, skill_config: SkillExtractionConfig) -> list[str]:
    """Extract the normalized skill set required or preferred by a job description."""
    return extract_document_skills(job_description, ner_model, skill_config)["hybrid_skills"]


def extract_resume_skills(resume_text: str, ner_model: Any, skill_config: SkillExtractionConfig) -> list[str]:
    """Extract the normalized skill set evidenced in a resume."""
    return extract_document_skills(resume_text, ner_model, skill_config)["hybrid_skills"]


def compare_skills(job_skills: list[str], resume_skills: list[str]) -> dict[str, list[str]]:
    """Return matched, missing, and additional skills in a stable alphabetical order."""
    job_set, resume_set = set(job_skills), set(resume_skills)
    return {
        "matched_skills": sorted(job_set & resume_set),
        "missing_skills": sorted(job_set - resume_set),
        "additional_resume_skills": sorted(resume_set - job_set),
    }


def calculate_semantic_similarity_score(
    job_description: str, resume_text: str, similarity_model: SentenceTransformer
) -> float:
    """Calculate a 0–100 cosine-similarity score for one job and one resume."""
    embeddings = similarity_model.encode([job_description, resume_text], show_progress_bar=False)
    cosine_score = float(cosine_similarity([embeddings[0]], [embeddings[1]])[0, 0])
    return round(max(0.0, min(1.0, cosine_score)) * 100, 1)


def calculate_skill_match_score(job_skills: list[str], resume_skills: list[str]) -> float | None:
    """Calculate job-skill coverage as a percentage; return None when no job skills are found."""
    if not job_skills:
        return None
    return round(len(set(job_skills) & set(resume_skills)) / len(set(job_skills)) * 100, 1)


def calculate_match_score(
    semantic_similarity_score: float, skill_match_score: float | None,
    requirement_match_score: float | None = None, config: ApplicationAnalysisConfig = APPLICATION_CONFIG
) -> dict[str, float | None]:
    """Combine available score components and renormalize weights for unavailable components."""
    components = {
        "semantic_similarity_score": (semantic_similarity_score, config.semantic_weight),
        "skill_match_score": (skill_match_score, config.skill_weight),
        "requirement_match_score": (requirement_match_score, config.requirement_weight),
    }
    available_weight = sum(weight for score, weight in components.values() if score is not None)
    if not available_weight:
        raise ValueError("At least one score component is required.")
    match_score = sum(score * weight for score, weight in components.values() if score is not None) / available_weight
    return {"match_score": round(match_score, 1), **{name: score for name, (score, _) in components.items()}}


def build_recommendations(skill_comparison: dict[str, list[str]], match_score: float, target_score: float) -> list[str]:
    """Create factual, prioritized recommendations without suggesting invented experience."""
    recommendations = []
    missing = skill_comparison["missing_skills"]
    if missing:
        recommendations.append(
            "Review these missing job skills and add them only when supported by real experience: " + ", ".join(missing)
        )
    if skill_comparison["matched_skills"]:
        recommendations.append(
            "Move relevant evidence for these matched skills closer to the professional summary and experience bullets: "
            + ", ".join(skill_comparison["matched_skills"])
        )
    if match_score < target_score:
        recommendations.append(
            f"The current score is below the {target_score:.0f}% target. Address real skill or experience gaps rather than adding unsupported claims."
        )
    if not recommendations:
        recommendations.append("No skill gaps were detected by the current extractors; review the semantic alignment of the resume language.")
    return recommendations

In [ ]:
def analyze_application(
    job_description: str, resume_text: str, target_score: float = 90.0,
    skill_config: SkillExtractionConfig = SKILL_CONFIG,
    analysis_config: ApplicationAnalysisConfig = APPLICATION_CONFIG,
    ner_model: Any | None = None, similarity_model: SentenceTransformer | None = None,
) -> dict[str, object]:
    """Analyze one user-provided job description and resume with explainable outputs."""
    if not job_description or not str(job_description).strip():
        raise ValueError("job_description must not be empty.")
    if not resume_text or not str(resume_text).strip():
        raise ValueError("resume_text must not be empty.")

    ner_model = ner_model or load_skill_ner_model(skill_config)
    similarity_model = similarity_model or SentenceTransformer(analysis_config.similarity_model_name)
    job_skills = extract_job_skills(job_description, ner_model, skill_config)
    resume_skills = extract_resume_skills(resume_text, ner_model, skill_config)
    skill_comparison = compare_skills(job_skills, resume_skills)
    semantic_score = calculate_semantic_similarity_score(job_description, resume_text, similarity_model)
    skill_score = calculate_skill_match_score(job_skills, resume_skills)
    score_breakdown = calculate_match_score(semantic_score, skill_score, config=analysis_config)

    return {
        **score_breakdown,
        "target_score": target_score,
        "job_skills": job_skills,
        "resume_skills": resume_skills,
        **skill_comparison,
        "recommendations": build_recommendations(skill_comparison, score_breakdown["match_score"], target_score),
    }


def display_application_analysis(analysis: dict[str, object]) -> None:
    """Print a compact, readable summary of a single application analysis."""
    print(f"JobLens Match Score: {analysis['match_score']}% (target: {analysis['target_score']}%)")
    print(f"Semantic similarity: {analysis['semantic_similarity_score']}%")
    print(f"Skill match: {analysis['skill_match_score']}%" if analysis["skill_match_score"] is not None else "Skill match: unavailable")
    print("\nMatched skills:", ", ".join(analysis["matched_skills"]) or "None detected")
    print("Missing skills:", ", ".join(analysis["missing_skills"]) or "None detected")
    print("\nRecommendations:")
    for recommendation in analysis["recommendations"]:
        print(f"- {recommendation}")

### Analyze user-provided text

In [ ]:
# Paste the job description and resume below, then uncomment the final two lines.
JOB_DESCRIPTION = """
Paste the job description here.
"""

RESUME_TEXT = """
Paste the resume text here.
"""

# application_analysis = analyze_application(JOB_DESCRIPTION, RESUME_TEXT)
# display_application_analysis(application_analysis)

## Part 4: Truthful Resume Tailoring

This section is provider-agnostic: supply a function that calls your chosen LLM. JobLens builds a strict prompt and validates the resulting draft before it is displayed. Never show an unvalidated generated resume as a final version.

In [ ]:
import re
from collections.abc import Callable


def build_resume_tailoring_prompt(
    job_description: str, original_resume: str, application_analysis: dict[str, object]
) -> str:
    """Build a fact-preserving prompt for a provider-specific resume rewriter."""
    matched_skills = ", ".join(application_analysis["matched_skills"]) or "None detected"
    missing_skills = ", ".join(application_analysis["missing_skills"]) or "None detected"
    return f"""You are revising a resume for a specific job.

Non-negotiable truthfulness rules:
- Use only facts, skills, employers, job titles, dates, credentials, projects, and achievements found in the original resume.
- Do not invent experience, qualifications, metrics, certifications, or proficiency.
- Do not add missing skills unless the original resume explicitly supports them.
- If a required skill is absent, omit it from the revised resume.

Matched skills to emphasize: {matched_skills}
Missing skills to avoid claiming: {missing_skills}

Job description:
{job_description}

Original resume:
{original_resume}

Return only the revised resume in a clear, ATS-friendly format.
"""


def generate_improved_resume(
    job_description: str, original_resume: str, application_analysis: dict[str, object],
    rewrite_resume: Callable[[str], str],
) -> str:
    """Generate a tailored resume through a caller-provided LLM function."""
    prompt = build_resume_tailoring_prompt(job_description, original_resume, application_analysis)
    revised_resume = rewrite_resume(prompt)
    if not revised_resume or not revised_resume.strip():
        raise ValueError("The resume rewriter returned an empty result.")
    return revised_resume.strip()


def extract_numeric_claims(text: str) -> set[str]:
    """Extract years, percentages, currency values, and count-based claims for review."""
    patterns = [r"\b(?:19|20)\d{2}\b", r"\b\d+(?:\.\d+)?%", r"[$€£]\s?\d[\d,]*(?:\.\d+)?", r"\b\d+\+?\s+(?:years?|months?|people|users|customers|projects?)\b"]
    return {match.lower() for pattern in patterns for match in re.findall(pattern, text, flags=re.IGNORECASE)}


def validate_resume_claims(
    original_resume: str, revised_resume: str, ner_model: Any, skill_config: SkillExtractionConfig
) -> dict[str, object]:
    """Flag new skills and quantitative claims that need the user’s confirmation."""
    original_skills = set(extract_resume_skills(original_resume, ner_model, skill_config))
    revised_skills = set(extract_resume_skills(revised_resume, ner_model, skill_config))
    new_skills = sorted(revised_skills - original_skills)
    new_numeric_claims = sorted(extract_numeric_claims(revised_resume) - extract_numeric_claims(original_resume))
    return {
        "is_safe_to_show": not new_skills and not new_numeric_claims,
        "new_skills_requiring_confirmation": new_skills,
        "new_numeric_claims_requiring_confirmation": new_numeric_claims,
    }

In [ ]:
def improve_and_reanalyze_resume(
    job_description: str, original_resume: str, rewrite_resume: Callable[[str], str],
    target_score: float = 90.0, skill_config: SkillExtractionConfig = SKILL_CONFIG,
    analysis_config: ApplicationAnalysisConfig = APPLICATION_CONFIG,
    ner_model: Any | None = None, similarity_model: SentenceTransformer | None = None,
) -> dict[str, object]:
    """Generate, validate, and conditionally score a truthful tailored-resume draft."""
    ner_model = ner_model or load_skill_ner_model(skill_config)
    similarity_model = similarity_model or SentenceTransformer(analysis_config.similarity_model_name)
    before = analyze_application(
        job_description, original_resume, target_score, skill_config, analysis_config, ner_model, similarity_model
    )
    tailored_resume = generate_improved_resume(job_description, original_resume, before, rewrite_resume)
    validation = validate_resume_claims(original_resume, tailored_resume, ner_model, skill_config)
    if not validation["is_safe_to_show"]:
        return {
            "before": before, "tailored_resume": tailored_resume, "validation": validation,
            "after": None,
            "message": "Review the flagged claims with the user before using or scoring this draft.",
        }

    after = analyze_application(
        job_description, tailored_resume, target_score, skill_config, analysis_config, ner_model, similarity_model
    )
    return {
        "before": before, "tailored_resume": tailored_resume, "validation": validation, "after": after,
        "target_reached": after["match_score"] >= target_score,
        "message": "Tailored resume validated and re-evaluated.",
    }


def display_before_after_comparison(result: dict[str, object]) -> None:
    """Display score changes and any validation action required after resume tailoring."""
    print(f"Before: {result['before']['match_score']}%")
    if result["after"] is None:
        print("After: not calculated because the draft needs claim review.")
        print("New skills requiring confirmation:", result["validation"]["new_skills_requiring_confirmation"])
        print("New numeric claims requiring confirmation:", result["validation"]["new_numeric_claims_requiring_confirmation"])
        return
    print(f"After: {result['after']['match_score']}%")
    print(f"Target reached: {result['target_reached']}")


# Example adapter: replace this function body with a call to the LLM provider you choose.
def rewrite_with_your_llm(prompt: str) -> str:
    """Call a configured LLM provider and return only the generated resume text."""
    raise NotImplementedError("Connect this adapter to an approved LLM provider before generating resumes.")

### Run safe tailoring

After implementing `rewrite_with_your_llm`, uncomment the following lines. The workflow stops before re-scoring whenever newly introduced skills or numeric claims require user confirmation.

In [ ]:
# tailoring_result = improve_and_reanalyze_resume(
#     JOB_DESCRIPTION, RESUME_TEXT, rewrite_with_your_llm
# )
# display_before_after_comparison(tailoring_result)